# Generate the first model based on feature importances provided by the Random Forest Regressor on 05_baseline_modeling

In [1]:
import os
import sys
from glob import glob

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import keras
import keras_tuner as kt
import tensorflow as tf
import tensorboard as tb
import numpy as np
import pandas as pd

from src.loaders import load_data_array
from utils import create_submission, get_run_logdir

2025-08-02 16:56:13.548755: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-02 16:56:13.744114: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754175373.816994     898 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754175373.837921     898 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1754175373.981144     898 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
def data_generator(sample_ids, features_subset):
    """Generator that yields one sample at a time"""
    wide_data = features_subset.values  # Your stats data
    
    for i, sample_id in enumerate(sample_ids):
        # Load one sample at a time
        X_deep_single, y_single = load_data_array([sample_id])
        
        # Convert to float32
        X_deep_single = X_deep_single[0].astype(np.float32)  # Remove batch dimension
        y_single = y_single[0].astype(np.float32)
        X_wide_single = wide_data[i].astype(np.float32)
        
        yield (X_wide_single, X_deep_single), y_single

features = pd.read_csv('../data/processed/final_features.csv')
is_train = features['split'] == 'train'

# Create datasets
train_ids = features.loc[is_train, 'sample_id'].reset_index(drop=True)
val_ids = features.loc[~is_train, 'sample_id'].reset_index(drop=True)

X_wide_train = features.loc[is_train].drop(columns=['sample_id', 'split', 'env_max', 'dom_freq'])
X_wide_val = features.loc[~is_train].drop(columns=['sample_id', 'split', 'env_max', 'dom_freq'])

# Create TensorFlow datasets
train_dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(train_ids, X_wide_train),
    output_signature=(
        (tf.TensorSpec(shape=(6,), dtype=tf.float32),           # wide input
         tf.TensorSpec(shape=(5, 10001, 31), dtype=tf.float32)), # deep input  
        tf.TensorSpec(shape=(300, 1259), dtype=tf.float32)      # target
    )
)

val_dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(val_ids, X_wide_val),
    output_signature=(
        (tf.TensorSpec(shape=(6,), dtype=tf.float32),
         tf.TensorSpec(shape=(5, 10001, 31), dtype=tf.float32)),
        tf.TensorSpec(shape=(300, 1259), dtype=tf.float32)
    )
)

# Optimize the pipeline
BATCH_SIZE = 8  # Start small due to large input size
train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("✅ TensorFlow datasets created!")

✅ TensorFlow datasets created!


I0000 00:00:1754175396.364786     898 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5563 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [3]:
# Inputs
input_wide = keras.layers.Input(shape = [6], name='stats')
input_deep = keras.layers.Input(shape = [5, 10001, 31], name = 'gathers')

input_deep_flatten = keras.layers.Flatten()(input_deep)

x = input_deep_flatten

# Add 3 layers, each with n_neurons
x = keras.layers.Dense(128, activation='relu')(x)
x = keras.layers.Dense(64, activation='relu')(x)
x = keras.layers.Dense(32, activation='relu')(x)

# Concatenation
concatenation_layer = keras.layers.Concatenate()([input_wide, x])

# Final dense layer
hidden4 = keras.layers.Dense(377700)(concatenation_layer)

# Output reshape
output = keras.layers.Reshape([300, 1259])(hidden4)

# Create the model
model = keras.Model(inputs = [input_wide, input_deep], outputs = output)

model.compile(
    optimizer='adamw',
    loss='mse',
    metrics = ['mape', 'mae']
)

In [4]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ gathers             │ (None, 5, 10001,  │          0 │ -                 │
│ (InputLayer)        │ 31)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 1550155)   │          0 │ gathers[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │ 198,419,9… │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stats (InputLayer)  │ (None, 6)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 38)        │          0 │ stats[0][0],      │
│ (Concatenate)       │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 377700)    │ 14,730,300 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 300, 1259) │          0 │ dense_3[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 213,160,604 (813.14 MB)

 Trainable params: 213,160,604 (813.14 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# Remember we need the following callbacks: tensoboard, reduce lr on plateau and model checkpoints
# Nevermind checkpoint will be used only when we get the best hyperparameters
log_dir = get_run_logdir(root_logdir='../logs/model02_mlp')

checkpoint_callback = keras.callbacks.ModelCheckpoint('../checkpoints/model_02_mlp.weights.h5', save_weights_only=True, save_best_only=True)
reduceLR_callback = keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5)
tensorboard_callback = keras.callbacks.TensorBoard(log_dir=log_dir)

In [6]:
model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    callbacks=[checkpoint_callback,reduceLR_callback,tensorboard_callback],   
)

Epoch 1/10


I0000 00:00:1754175409.334770    1120 service.cc:152] XLA service 0x7fb3e801d1f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1754175409.334801    1120 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2025-08-02 16:56:49.378276: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1754175409.531310    1120 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-08-02 16:56:51.093361: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_402_0', 96 bytes spill stores, 96 bytes spill loads



      2/Unknown 6s 57ms/step - loss: 8.0153 - mae: 2.6695 - mape: 99.9298

I0000 00:00:1754175412.125968    1120 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


    200/Unknown 296s 1s/step - loss: 1.3237 - mae: 0.6869 - mape: 26.3941

...

200/200 ━━━━━━━━━━━━━━━━━━━━ 363s 2s/step - loss: 0.0520 - mae: 0.1555 - mape: 6.0079 - val_loss: 0.0526 - val_mae: 0.1635 - val_mape: 6.3904 - learning_rate: 0.0010
Epoch 9/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.0544 - mae: 0.1592 - mape: 6.1677

2025-08-02 17:50:08.565950: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 12952693542331411899
2025-08-02 17:50:08.565988: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 9613141335965365486
2025-08-02 17:50:08.566003: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 14010388118964426356
2025-08-02 17:51:17.230551: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 12952693542331411899
2025-08-02 17:51:17.230605: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 9613141335965365486
2025-08-02 17:51:17.230644: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 14010388118964426356


200/200 ━━━━━━━━━━━━━━━━━━━━ 362s 2s/step - loss: 0.0512 - mae: 0.1542 - mape: 5.9404 - val_loss: 0.0524 - val_mae: 0.1646 - val_mape: 6.4282 - learning_rate: 0.0010
Epoch 10/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.0548 - mae: 0.1608 - mape: 6.2249

2025-08-02 17:56:05.021043: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 12952693542331411899
2025-08-02 17:56:05.021087: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 14010388118964426356
2025-08-02 17:57:15.423194: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 12952693542331411899
2025-08-02 17:57:15.423228: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 9613141335965365486
2025-08-02 17:57:15.423242: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 14010388118964426356


200/200 ━━━━━━━━━━━━━━━━━━━━ 358s 2s/step - loss: 0.0519 - mae: 0.1565 - mape: 6.0335 - val_loss: 0.0517 - val_mae: 0.1634 - val_mape: 6.3811 - learning_rate: 0.0010


In [7]:
# Use the model to make predictions on the test set.
model.save('../models/model_02_mlp.keras')  # Saves architecture + weights
print("✅ Model saved to ../models/model_02_mlp.keras")

✅ Model saved to ../models/model_02_mlp.keras
